# Non-AI Baselines and the Colorado Pair
## ACL ARR 2026: The Language of AI Liability

Is a pattern specific to AI regulation, or a drafting norm of the jurisdiction?
Each AI law is compared with one recent non-AI technology law from the same
jurisdiction. Colorado SB 24-205 is also compared with its successor SB 26-189,
parallel to California SB 1047 and SB 53.

Run `Data_Preprocessing.ipynb` first: it writes `baselines_corpus.json`.
All measures and tests reuse the definitions of the technique notebooks.

In [ ]:
# Run once per environment (Colab or local)
import sys
%pip install -q spacy spacy-curated-transformers
!{sys.executable} -m spacy download en_core_web_trf

In [2]:
# ── Cell 1: Setup and load ──

import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
from scipy.stats import chi2_contingency, fisher_exact

BASE_DIR = Path("Data")
OUTPUT_DIR = BASE_DIR / "processed"
results = json.load(open(OUTPUT_DIR / "preprocessed_corpus.json", encoding="utf-8"))
results.update(json.load(open(OUTPUT_DIR / "baselines_corpus.json", encoding="utf-8")))

spacy.prefer_gpu()  # uses a GPU when one is available
nlp = spacy.load("en_core_web_trf")
nlp.max_length = 700_000

PAIRS = [
    ("EU: AI Act vs DSA",                  "eu_ai_act",   "eu_dsa"),
    ("China: GenAI Measures vs PIPL",      "china_genai", "cn_pipl"),
    ("S. Korea: AI Basic Act vs PIPA",     "south_korea", "kr_pipa"),
    ("Colorado: SB 24-205 vs Privacy Act", "co_sb205",    "co_sb21_190"),
    ("Colorado: SB 24-205 vs SB 26-189",   "co_sb205",    "co_sb26_189"),
]
LAWS = sorted({law for _, a, b in PAIRS for law in (a, b)})
print(f"Loaded {len(results)} laws; comparing {len(PAIRS)} pairs")


Loaded 15 laws; comparing 5 pairs


In [3]:
# ── Cell 2: Measures and tests (same definitions as the technique notebooks) ──

# Technique 2: voice (Responsibility_Attribution_Analysis, cell 2)
def classify_voice(doc):
    """Classify a parsed sentence as active, passive, or indeterminate
    based on dependency labels.

    Returns:
      'active'  — has nsubj (nominal subject, active)
      'passive' — has nsubjpass or has auxpass
      'indeterminate' — neither detected (e.g., fragments, imperatives)
    """
    has_nsubj = False
    has_passive = False

    for token in doc:
        if token.dep_ == "nsubj":
            has_nsubj = True
        if token.dep_ in ("nsubjpass", "nsubj:pass"):
            has_passive = True
        if token.dep_ == "auxpass" or token.dep_ == "aux:pass":
            has_passive = True

    if has_passive:
        return "passive"
    elif has_nsubj:
        return "active"
    else:
        return "indeterminate"


# Technique 2: actor classification (Responsibility_Attribution_Analysis, cells 3-4)
SEED_ACTORS = {
    "regulated_entity": [
        "developer", "provider", "deployer", "operator", "distributor",
        "importer", "manufacturer", "authorised representative", "employer",
        "covered entity", "service provider", "responsible person",
        "data controller", "processor", "vendor", "company", "business",
        "organization",
    ],
    "government": [
        "commission", "agency", "authority", "office", "board", "committee",
        "department", "ministry", "attorney general", "secretary", "government",
        "member state", "notified body", "market surveillance",
        "national authority", "competent authority", "advisory body",
        "council", "minister", "state",
    ],
    "generic": [
        "person", "entity", "individual", "one", "natural person",
        "legal person", "user", "applicant", "complainant", "affected person",
        "third party", "public",
    ],
}
TERM_CATEGORY = {t: c for c, ts in SEED_ACTORS.items() for t in ts}

# Spelling variants folded onto one seed term
VARIANTS = {"authorized": "authorised", "organisation": "organization",
            "governmental": "government"}
# A phrase containing one of these modifiers is government / institutional
GOV_MODIFIERS = {"government", "market surveillance"}
# Anaphoric subjects: a fixed pronoun list (spaCy's PRON tag also fires on
# list markers such as "(a)", so it is not used)
PRONOUNS = {"it", "they", "he", "she", "which", "who", "that", "this",
            "these", "those", "we", "you", "i", "them", "its", "their"}


def singular(word):
    if word.endswith("ies") and len(word) > 4:
        return word[:-3] + "y"          # bodies -> body
    if word.endswith("ses") or word.endswith("xes"):
        return word[:-2]                # businesses -> business
    if word.endswith("s") and not word.endswith("ss"):
        return word[:-1]                # providers -> provider
    return word


def classify_phrase(words):
    """Classify a head noun phrase (lower-case words, head noun last).

    Only seed terms that END with the head noun are candidates, so the head
    decides the actor type; the longest candidate wins ("member state" over
    "state"). Returns (seed_term, category) or (None, "unclassified").
    """
    words = [w for w in words if re.fullmatch(r"[a-z][a-z\-']*", w)]
    if not words:
        return None, "unclassified"
    words = words[:-1] + [singular(words[-1])]
    phrase = " ".join(VARIANTS.get(w, w) for w in words)
    best = None
    for term, cat in TERM_CATEGORY.items():
        t = term.split()
        if phrase.split()[-len(t):] == t and (best is None or len(t) > len(best[0].split())):
            best = (term, cat)
    if best is None:
        return None, "unclassified"
    term, cat = best
    if cat != "government" and any(f" {m} " in f" {phrase} " for m in GOV_MODIFIERS):
        return term, "government"
    return term, cat


def extract_subject(doc):
    """Extract the subject noun phrase from a parsed sentence.
    Returns the text of the subject's subtree (the full noun phrase)
    and the head noun."""
    for token in doc:
        if token.dep_ == "nsubj":
            # Get the full noun phrase via the subtree
            subtree_span = doc[token.left_edge.i : token.right_edge.i + 1]
            return {
                "head": token.text.lower(),
                "full_np": subtree_span.text,
                "head_lemma": token.lemma_.lower(),
            }
    return None


def head_phrase(doc):
    """Head noun of the subject that extract_subject uses, plus its compound
    and adjectival modifiers, as lower-case words in surface order."""
    for token in doc:
        if token.dep_ == "nsubj":
            mods, stack = [], [c for c in token.lefts if c.dep_ in ("compound", "amod")]
            while stack:
                c = stack.pop()
                mods.append(c)
                stack.extend(cc for cc in c.lefts if cc.dep_ in ("compound", "amod"))
            return [t.text.lower() for t in sorted(mods, key=lambda t: t.i) + [token]]
    return []


def classify_actor(words):
    """(seed_term, category) for a subject head phrase."""
    if not words:
        return None, "unclassified"
    if len(words) == 1 and words[0] in PRONOUNS:
        return words[0], "anaphoric"
    return classify_phrase(words)



# Technique 3: tree depth and conditional markers (Obligation_Complexity_Analysis)
def tree_depth(token):
    """Compute depth of a token in the dependency tree.
    Root has depth 0."""
    depth = 0
    while token.head != token:
        depth += 1
        token = token.head
    return depth


def max_tree_depth(doc):
    """Maximum dependency tree depth across all tokens in a doc."""
    if len(doc) == 0:
        return 0
    return max(tree_depth(tok) for tok in doc)

CONDITIONAL_MARKERS = re.compile(
    r'\bif\b'
    r'|\bunless\b'
    r'|\bprovided\s+that\b'
    r'|\bexcept\s+(?:when|where|that|as)\b'
    r'|\bto\s+the\s+extent\s+that\b'
    r'|\bsubject\s+to\b'
    r'|\bnotwithstanding\b'
    r'|\bin\s+the\s+event\s+that\b'
    r'|\bwhere\b'
    r'|\bin\s+(?:the\s+)?case\s+(?:of|that|where)\b'
    r'|\bprovided\b',
    re.IGNORECASE
)

# Technique 4: hedge terms (Regulatory_Specificity_Index)
HEDGE_WORDS = re.compile(
    r'\breasonable\b'
    r'|\breasonably\b'
    r'|\bappropriate\b'
    r'|\bappropriately\b'
    r'|\badequate\b'
    r'|\badequately\b'
    r'|\bsufficient\b'
    r'|\bsufficiently\b'
    r'|\bmaterial\b'
    r'|\bmaterially\b'
    r'|\bsignificant\b'
    r'|\bsignificantly\b'
    r'|\bas\s+applicable\b'
    r'|\bas\s+necessary\b'
    r'|\bas\s+appropriate\b'
    r'|\bto\s+the\s+extent\s+practicable\b'
    r'|\bto\s+the\s+extent\s+possible\b'
    r'|\bwhere\s+appropriate\b'
    r'|\bwhere\s+applicable\b'
    r'|\bwhere\s+necessary\b'
    r'|\bproportionate\b'
    r'|\bsuitable\b'
    r'|\bfeasible\b',
    re.IGNORECASE
)

# Statistical tests (Deontic_Modal_Analysis, Obligation_Complexity_Analysis)
def cramers_v(contingency_table):
    """Compute Cramér's V from a contingency table."""
    chi2, p, dof, _ = chi2_contingency(contingency_table)
    n = contingency_table.sum()
    k = min(contingency_table.shape) - 1
    if k == 0 or n == 0:
        return 0.0
    return np.sqrt(chi2 / (n * k))


def permutation_test(group_a, group_b, n_permutations=10000, seed=42):
    """Two-sided permutation test on difference in means."""
    rng = np.random.default_rng(seed)
    observed_diff = abs(np.mean(group_a) - np.mean(group_b))
    combined = np.concatenate([group_a, group_b])
    n_a = len(group_a)

    count_extreme = 0
    for _ in range(n_permutations):
        perm = rng.permutation(combined)
        perm_diff = abs(np.mean(perm[:n_a]) - np.mean(perm[n_a:]))
        if perm_diff >= observed_diff:
            count_extreme += 1

    p_value = (count_extreme + 1) / (n_permutations + 1)
    return observed_diff, p_value


def cohens_d(group_a, group_b):
    """Compute Cohen's d for two groups."""
    n_a, n_b = len(group_a), len(group_b)
    if n_a < 2 or n_b < 2:
        return 0.0
    var_a, var_b = np.var(group_a, ddof=1), np.var(group_b, ddof=1)
    pooled_std = np.sqrt(((n_a - 1) * var_a + (n_b - 1) * var_b) / (n_a + n_b - 2))
    if pooled_std == 0:
        return 0.0
    return abs(np.mean(group_a) - np.mean(group_b)) / pooled_std


In [4]:
# ── Cell 3: Per-law measures ──

CATS4 = ["HARD_OBLIGATION", "SOFT_OBLIGATION", "PERMISSION", "PROHIBITION"]
REAL = ["regulated_entity", "government", "generic"]

measures = {}
for law_id in LAWS:
    oblig = [s["text"] for s in results[law_id]["sentences"] if s["is_obligation"]]
    docs = list(nlp.pipe(oblig, batch_size=16))
    voice = Counter(classify_voice(d) for d in docs)
    actors = Counter(classify_actor(head_phrase(d))[1] for d in docs if classify_voice(d) == "active")
    measures[law_id] = {
        "modal": [results[law_id]["stats"]["deontic_counts"].get(c, 0) for c in CATS4],
        "voice": {k: voice.get(k, 0) for k in ("active", "passive", "indeterminate")},
        "actors": [actors.get(c, 0) for c in REAL],
        "depth": np.array([max_tree_depth(d) for d in docs]),
        "conditional": np.array([len(CONDITIONAL_MARKERS.findall(t)) for t in oblig]),
        "hedge": np.array([len(HEDGE_WORDS.findall(t)) for t in oblig]),
    }
    m = measures[law_id]
    print(f"{law_id:<12} n={len(oblig):>3}  active={100 * m['voice']['active'] / len(oblig):3.0f}%  "
          f"actors={m['actors']}  depth={m['depth'].mean():.2f}  "
          f"cond={m['conditional'].mean():.2f}  hedge={m['hedge'].mean():.2f}")


china_genai  n= 16  active= 75%  actors=[9, 0, 0]  depth=9.81  cond=0.44  hedge=0.06


cn_pipl      n= 78  active= 59%  actors=[2, 3, 5]  depth=8.31  cond=1.04  hedge=0.14


co_sb205     n= 25  active= 56%  actors=[8, 3, 0]  depth=11.80  cond=1.84  hedge=1.24


co_sb21_190  n= 65  active= 75%  actors=[1, 5, 0]  depth=8.72  cond=0.60  hedge=0.46


co_sb26_189  n= 35  active= 51%  actors=[8, 4, 0]  depth=9.03  cond=1.14  hedge=0.97


eu_ai_act    n=710  active= 53%  actors=[59, 163, 1]  depth=9.38  cond=0.59  hedge=0.39


eu_dsa       n=552  active= 71%  actors=[65, 141, 0]  depth=9.91  cond=0.51  hedge=0.27


kr_pipa      n=419  active= 50%  actors=[3, 60, 12]  depth=9.86  cond=1.06  hedge=0.06


south_korea  n=102  active= 58%  actors=[3, 38, 2]  depth=9.88  cond=0.37  hedge=0.12


In [5]:
# ── Cell 4: Pairwise tests ──

def category_test(row_a, row_b):
    table = np.array([row_a, row_b])
    table = table[:, table.sum(axis=0) > 0]
    if table.shape[1] < 2:
        return {"test": "n/a"}
    if table.shape[1] == 2:
        odds_ratio, p = fisher_exact(table)
        return {"test": "Fisher", "stat": float(odds_ratio), "p": float(p)}
    chi2, p, dof, expected = chi2_contingency(table)
    return {"test": "chi2", "stat": float(chi2), "p": float(p),
            "V": float(cramers_v(table)), "min_exp": float(expected.min())}


def numeric_test(a, b):
    diff, p = permutation_test(a, b)
    return {"a": float(a.mean()), "b": float(b.mean()), "p": float(p), "d": float(cohens_d(a, b))}


out = []
for name, a, b in PAIRS:
    ma, mb = measures[a], measures[b]
    va, vb = ma["voice"], mb["voice"]
    odds_ratio, p_voice = fisher_exact([[va["active"], va["passive"]], [vb["active"], vb["passive"]]])
    out.append({
        "pair": name, "a": a, "b": b,
        "modal": category_test(ma["modal"], mb["modal"]),
        "voice": {"OR": float(odds_ratio), "p": float(p_voice)},
        "active_pct": [va["active"] / sum(va.values()), vb["active"] / sum(vb.values())],
        "actors": category_test(ma["actors"], mb["actors"]),
        "actor_counts": [ma["actors"], mb["actors"]],
        **{k: numeric_test(ma[k], mb[k]) for k in ("depth", "conditional", "hedge")},
    })

with open(OUTPUT_DIR / "baseline_tests.json", "w", encoding="utf-8") as f:
    json.dump(out, f, indent=1)

fmt = lambda t: "n/a" if t.get("test") == "n/a" else f"{t['test']} p={t['p']:.3f}"
rows = [{"pair": r["pair"], "modal": fmt(r["modal"]), "voice p": round(r["voice"]["p"], 3),
         "actors": fmt(r["actors"]),
         **{k: f"{r[k]['a']:.2f} vs {r[k]['b']:.2f} (p={r[k]['p']:.3f}, d={r[k]['d']:.2f})"
            for k in ("depth", "conditional", "hedge")}} for r in out]
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nSaved: {OUTPUT_DIR / 'baseline_tests.json'}")


                              pair        modal  voice p         actors                           depth                    conditional                          hedge
                 EU: AI Act vs DSA chi2 p=0.095    0.000   chi2 p=0.331  9.38 vs 9.91 (p=0.062, d=0.11) 0.59 vs 0.51 (p=0.192, d=0.07) 0.39 vs 0.27 (p=0.005, d=0.16)
     China: GenAI Measures vs PIPL chi2 p=0.040    0.271   chi2 p=0.002  9.81 vs 8.31 (p=0.109, d=0.46) 0.44 vs 1.04 (p=0.071, d=0.52) 0.06 vs 0.14 (p=0.776, d=0.12)
    S. Korea: AI Basic Act vs PIPA chi2 p=0.000    0.187   chi2 p=0.159  9.88 vs 9.86 (p=0.965, d=0.00) 0.37 vs 1.06 (p=0.001, d=0.39) 0.12 vs 0.06 (p=0.081, d=0.20)
Colorado: SB 24-205 vs Privacy Act chi2 p=0.133    0.122 Fisher p=0.050 11.80 vs 8.72 (p=0.000, d=0.82) 1.84 vs 0.60 (p=0.001, d=0.83) 1.24 vs 0.46 (p=0.025, d=0.53)
  Colorado: SB 24-205 vs SB 26-189 chi2 p=0.573    1.000 Fisher p=1.000 11.80 vs 9.03 (p=0.019, d=0.62) 1.84 vs 1.14 (p=0.146, d=0.39) 1.24 vs 0.97 (p=0.641, d=0.15)

Sav